# Lecture16: ML/RL ??? ??? KuCoin (Google Colab)

??????? ???????: ???????? OHLCV ? KuCoin, ??????? `ret_hat` ????? ARIMA, `sigma_hat` ????? GARCH,
????????? `latest_forecast_signal_kucoin_rl.json` ? ????????? RL-??????????? ?? ?????????.

## 1. ????????? ?????????

In [ ]:
!pip -q install pandas numpy requests statsmodels arch matplotlib

## 2. ???????? ???????? ?????? KuCoin (REST OHLCV)

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from statsmodels.tsa.arima.model import ARIMA
from arch import arch_model

SPOT_SYMBOL = 'NEAR-USDT'
CANDLE_TYPE = '1min'
LIMIT = 1200

url = 'https://api.kucoin.com/api/v1/market/candles'
params = {'symbol': SPOT_SYMBOL, 'type': CANDLE_TYPE}
raw = requests.get(url, params=params, timeout=20).json()['data']

df = pd.DataFrame(raw, columns=['ts', 'open', 'close', 'high', 'low', 'volume', 'turnover'])
df = df.iloc[::-1].reset_index(drop=True)
for c in ['open', 'close', 'high', 'low', 'volume', 'turnover']:
    df[c] = df[c].astype(float)
df['ts'] = pd.to_datetime(df['ts'].astype(int), unit='s', utc=True)
df = df.tail(LIMIT).copy()
df.tail(3)

## 3. ML-????: ARIMA ??? ?????????? ? GARCH ??? ?????????????

In [ ]:
df['ret'] = np.log(df['close']).diff()
ret_series = df['ret'].dropna()

arima = ARIMA(ret_series, order=(1, 0, 1)).fit()
ret_hat = float(arima.forecast(1).iloc[0])

garch = arch_model(ret_series * 100, mean='Zero', vol='GARCH', p=1, q=1).fit(disp='off')
var_1 = float(garch.forecast(horizon=1).variance.iloc[-1, 0])
sigma_hat = (var_1 ** 0.5) / 100

z = ret_hat / max(sigma_hat, 1e-8)
ret_hat, sigma_hat, z

## 4. ???????? ?????????? ????????? ??? RL-????

In [ ]:
spot_price = float(df['close'].iloc[-1])
futures_price = spot_price

state = {
    'timestamp': df['ts'].iloc[-1].isoformat(),
    'asset': 'NEAR',
    'spot_symbol': 'NEAR-USDT',
    'futures_symbol': 'NEARUSDTM',
    'spot_price': spot_price,
    'futures_price': futures_price,
    'ret_hat': ret_hat,
    'sigma_hat': sigma_hat,
    'z_score': z,
    'candle_type': '1min'
}

out_dir = Path('reports/kucoin_rl')
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / 'latest_forecast_signal_kucoin_rl.json'
out_path.write_text(json.dumps(state, ensure_ascii=False, indent=2), encoding='utf-8')

print('JSON ????????:', out_path)
state

## 5. ??????? ??? ????????? (??? ? lecture15)

??????? ???? ??????????? ? ????? ??????????? `lecture16`.

In [ ]:
print('SHADOW ???????? ??????:')
print('python run_trade_signal.py --mode shadow --config config/micro_near_v1_1m.json --state-json reports/kucoin_rl/latest_forecast_signal_kucoin_rl.json')
print()
print('LIVE ???????? ??????:')
print('python run_trade_signal.py --run-real-order --config config/micro_near_v1_1m.json --state-json reports/kucoin_rl/latest_forecast_signal_kucoin_rl.json')
print()
print('?????? ??????? (??? ??????):')
print('python run_trade_signal.py --run-real-order --config config/micro_near_v1_1m.json --state-json reports/kucoin_rl/latest_forecast_signal_kucoin_rl.json --force-action BUY_BOTH --spot-qty 0.1 --futures-contracts 1')
print('python run_trade_signal.py --run-real-order --config config/micro_near_v1_1m.json --state-json reports/kucoin_rl/latest_forecast_signal_kucoin_rl.json --force-action SELL_BOTH --spot-qty 0.1 --futures-contracts 1')
print()
print('?????? ??????? ?? ???????????:')
print('python run_trade_signal.py --run-real-order --config config/micro_near_v1_1m.json --state-json reports/kucoin_rl/latest_forecast_signal_kucoin_rl.json --force-action BUY_SPOT --spot-qty 0.1')
print('python run_trade_signal.py --run-real-order --config config/micro_near_v1_1m.json --state-json reports/kucoin_rl/latest_forecast_signal_kucoin_rl.json --force-action SELL_SPOT --spot-qty 0.1')
print('python run_trade_signal.py --run-real-order --config config/micro_near_v1_1m.json --state-json reports/kucoin_rl/latest_forecast_signal_kucoin_rl.json --force-action BUY_FUTURES --futures-contracts 1')
print('python run_trade_signal.py --run-real-order --config config/micro_near_v1_1m.json --state-json reports/kucoin_rl/latest_forecast_signal_kucoin_rl.json --force-action SELL_FUTURES --futures-contracts 1')

## 6. ???????? ???? ???? (??????)

??? ????????? ????? ????????? ??? ? ?????? ? ?? ?????? ???? ????? ???????, ?????????, ??????? ??? ??????????? ???????.

In [ ]:
import os
import time

for i in range(3):
    print(f'???????? {i + 1}/3')
    os.system('python run_trade_signal.py --mode shadow --config config/micro_near_v1_1m.json --state-json reports/kucoin_rl/latest_forecast_signal_kucoin_rl.json')
    time.sleep(60)